# Imports & Setup

In [17]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch
from torch import nn
import matplotlib.pyplot as plt
import os
import numpy as np
import imageio
from collections import OrderedDict
from typing import Dict, Callable
from IPython.display import display, Markdown

# Load Model and Tokenizer

In [2]:
model_name = "Qwen/Qwen3-4B"
device = 'cuda' if torch.cuda.is_available() else 'cpu'
device

'cuda'

In [3]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(model_name)
print(tokenizer)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

Qwen2TokenizerFast(name_or_path='Qwen/Qwen3-4B', vocab_size=151643, model_max_length=131072, is_fast=True, padding_side='right', truncation_side='right', special_tokens={'eos_token': '<|im_end|>', 'pad_token': '<|endoftext|>', 'additional_special_tokens': ['<|im_start|>', '<|im_end|>', '<|object_ref_start|>', '<|object_ref_end|>', '<|box_start|>', '<|box_end|>', '<|quad_start|>', '<|quad_end|>', '<|vision_start|>', '<|vision_end|>', '<|vision_pad|>', '<|image_pad|>', '<|video_pad|>']}, clean_up_tokenization_spaces=False, added_tokens_decoder={
	151643: AddedToken("<|endoftext|>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	151644: AddedToken("<|im_start|>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	151645: AddedToken("<|im_end|>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	151646: AddedToken("<|object_ref_start|>", rstrip=False, lstrip=False, single_word=False, normalized=F

In [4]:
from transformers import AutoModelForCausalLM

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    dtype='auto',
    device_map=device
)

print(model.device)
print(model)

config.json:   0%|          | 0.00/726 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

model-00002-of-00003.safetensors:   0%|          | 0.00/3.99G [00:00<?, ?B/s]

model-00003-of-00003.safetensors:   0%|          | 0.00/99.6M [00:00<?, ?B/s]

model-00001-of-00003.safetensors:   0%|          | 0.00/3.96G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

cuda:0
Qwen3ForCausalLM(
  (model): Qwen3Model(
    (embed_tokens): Embedding(151936, 2560)
    (layers): ModuleList(
      (0-35): 36 x Qwen3DecoderLayer(
        (self_attn): Qwen3Attention(
          (q_proj): Linear(in_features=2560, out_features=4096, bias=False)
          (k_proj): Linear(in_features=2560, out_features=1024, bias=False)
          (v_proj): Linear(in_features=2560, out_features=1024, bias=False)
          (o_proj): Linear(in_features=4096, out_features=2560, bias=False)
          (q_norm): Qwen3RMSNorm((128,), eps=1e-06)
          (k_norm): Qwen3RMSNorm((128,), eps=1e-06)
        )
        (mlp): Qwen3MLP(
          (gate_proj): Linear(in_features=2560, out_features=9728, bias=False)
          (up_proj): Linear(in_features=2560, out_features=9728, bias=False)
          (down_proj): Linear(in_features=9728, out_features=2560, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): Qwen3RMSNorm((2560,), eps=1e-06)
        (post_attentio

In [10]:
# Apply chat template to prompt
prompt = "Solve this equation for x: 2x + 5 = 17. Please reason step by step, and put your final answer within \boxed{}."
messages = [
    {"role": "user", "content": prompt}
]
tokenized_chat = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
    enable_thinking=True # Switches between thinking and non-thinking modes. Default is True.
)
print(tokenized_chat)

# Tokenize inputs and move to device
inputs = tokenizer([tokenized_chat], return_tensors="pt").to(model.device)
print(inputs)

<|im_start|>user
Solve this equation for x: 2x + 5 = 17<|im_end|>
<|im_start|>assistant

{'input_ids': tensor([[151644,    872,    198,     50,   3948,    419,  23606,    369,    856,
             25,    220,     17,     87,    488,    220,     20,    284,    220,
             16,     22, 151645,    198, 151644,  77091,    198]],
       device='cuda:0'), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1]], device='cuda:0')}


In [19]:
generated_ids = model.generate(
    **inputs,
    max_new_tokens=2048,
    temperature=0.6,
    top_p=0.95,
    top_k=20
)

output_ids = generated_ids[0][len(inputs.input_ids[0]):].tolist()
response = tokenizer.decode(output_ids, skip_special_tokens=True)

print(response)

<think>
Okay, so I need to solve the equation 2x + 5 = 17 for x. Hmm, let me think. I remember from algebra that the goal is to isolate x on one side of the equation. Let me start by writing down the equation again to make sure I have it right: 2x + 5 = 17.

First, I think I need to get rid of that 5 that's being added to the 2x. To do that, I can subtract 5 from both sides of the equation. That way, I keep the equation balanced. Let me try that. So if I subtract 5 from both sides, it would look like:

2x + 5 - 5 = 17 - 5

Simplifying both sides, the +5 and -5 on the left side cancel each other out, leaving just 2x. On the right side, 17 minus 5 is 12. So now the equation is 2x = 12. Okay, that seems right.

Now, I need to get x by itself. Since 2 is multiplied by x, I should divide both sides by 2 to undo that multiplication. Let me do that. Dividing both sides by 2 gives:

2x / 2 = 12 / 2

Simplifying that, the 2's on the left side cancel out, leaving x. And 12 divided by 2 is 6. So 